<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6: PyTorch</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: keep -->
<div style='color:rgb(22,60,105); font-size:1.3em; font-weight:bold; border-left: 5px solid rgb(255,106,0); padding-left:0.8em; margin-top:1em;'>Where We Are</div>

Sessions 2–5 built everything **from scratch**:
- forward pass: linear layers, activations, loss functions
- backward pass: local and global gradients, backpropagation through a network
- optimisation: gradient descent, mini-batch SGD, momentum, Adam

All using raw PyTorch tensors — no `torch.nn`, no `torch.optim`.

**This session:** learn the PyTorch API — the tools practitioners use to build and train networks.

<!-- FULL: keep, DEMO: keep -->
<div style='color:rgb(22,60,105); font-size:1.3em; font-weight:bold; border-left: 5px solid rgb(255,106,0); padding-left:0.8em; margin-top:1em;'>Session Overview</div>

1. Tensors & autograd
2. Building networks — `nn.Module`
3. Optimisers — `torch.optim`
4. Data pipeline — `Dataset` & `DataLoader`
5. Full training loop
6. Going deeper

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>1</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Tensors &amp; Autograd</span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Tensors
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch's fundamental data structure — an $n$-dimensional array.

`requires_grad=True` tells PyTorch to track all operations on a tensor 
so gradients can be computed via `.backward()`.

In [1]:
import torch

# ordinary tensor — no gradient tracking
x = torch.tensor([2.0, 3.0])
print(f'x:               {x}')
print(f'requires_grad:   {x.requires_grad}')
print(f'grad_fn:         {x.grad_fn}')

x:               tensor([2., 3.])
requires_grad:   False
grad_fn:         None


In [2]:
# tensor with gradient tracking
theta = torch.tensor([1.0, 0.5], requires_grad=True)
print(f'theta:           {theta}')
print(f'requires_grad:   {theta.requires_grad}')
print(f'grad_fn:         {theta.grad_fn}')   # None — leaf node

theta:           tensor([1.0000, 0.5000], requires_grad=True)
requires_grad:   True
grad_fn:         None


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; creating leaf tensors with requires_grad
</div>

<!-- FULL: keep, DEMO: delete -->
Three equivalent ways to create a trainable parameter tensor:

In [3]:
import torch.nn as nn   # needed for nn.Parameter below

# Option 1 — torch.tensor with requires_grad=True (from Python data)
theta_a = torch.tensor([1.0, 0.5], requires_grad=True)

# Option 2 — in-place requires_grad_ on an existing tensor
theta_b = torch.zeros(2).requires_grad_(True)

# Option 3 — nn.Parameter (preferred inside nn.Module)
theta_c = nn.Parameter(torch.tensor([1.0, 0.5]))

for name, t in [('tensor + flag', theta_a),
                 ('zeros + requires_grad_', theta_b),
                 ('nn.Parameter', theta_c)]:
    print(f'{name:30s}  requires_grad={t.requires_grad}')

tensor + flag                   requires_grad=True
zeros + requires_grad_          requires_grad=True
nn.Parameter                    requires_grad=True


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; creating tensor from another tensor
</div>

<!-- FULL: keep, DEMO: delete -->
Passing an existing tensor to `torch.tensor()` with `requires_grad=True` raises a `RuntimeError`.

The fix is `.clone().detach().requires_grad_(True)` — two operations for two distinct reasons:

- **`.clone()`** — copies the data so the new tensor has independent storage; without it, both tensors share the same memory and modifying one silently changes the other
- **`.detach()`** — cuts the new tensor out of any existing computation graph; without it, gradients would try to flow back through `x_existing`'s history during `.backward()`

You need both: `.clone()` for data independence, `.detach()` for graph independence. When `x_existing` is a fresh `torch.randn()` with no `grad_fn`, `.detach()` alone would suffice — but `.clone().detach()` is the safe idiomatic form that works in all cases.

In [4]:
x_existing = torch.randn(3)

# Wrong — raises RuntimeError
theta_bad = torch.tensor(x_existing, requires_grad=True)

/tmp/ipykernel_17668/642991285.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  theta_bad = torch.tensor(x_existing, requires_grad=True)


In [5]:
# Correct
theta_ok = x_existing.clone().detach().requires_grad_(True)
print(f'theta_ok: {theta_ok}')
print(f'requires_grad: {theta_ok.requires_grad}')

theta_ok: tensor([ 2.0562, -1.0287,  0.6386], requires_grad=True)
requires_grad: True


<!-- FULL: keep, DEMO: delete -->
**Why `.clone()` matters — shared storage demo:**

Without `.clone()`, two tensors share the same underlying memory. Modifying one silently changes the other.

In [6]:
x = torch.tensor([1.0, 2.0, 3.0])

# without clone — shared storage
y = x.detach()          # no copy, same memory
y[0] = 99.0
print(f'x after modifying y (no clone): {x}')  # x is also changed!

# with clone — independent storage
x = torch.tensor([1.0, 2.0, 3.0])   # reset
z = x.clone().detach()
z[0] = 99.0
print(f'x after modifying z (clone):    {x}')  # x unchanged

x after modifying y (no clone): tensor([99.,  2.,  3.])
x after modifying z (clone):    tensor([1., 2., 3.])


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Dynamic Computation Graph
</div>

<!-- FULL: keep, DEMO: delete -->
Every operation on a `requires_grad` tensor is recorded in a 
**dynamic computation graph**. Each result gets a `grad_fn` — 
a reference to the operation that created it. 
This is the graph traversed during `.backward()`.

<!-- FULL: keep, DEMO: delete -->
Recall from Session 4 — the computation graph has three types of nodes:

<img src='../../Common/Pics/CompGraph_1.svg' style='max-width:70%; margin:1em auto; display:block;'/>

- **Input nodes** — data fed into the network; no gradient tracked
- **Parameter nodes** — learnable weights and biases; `requires_grad=True`, gradient accumulates in `.grad`
- **Compute nodes** — results of operations; each has a `grad_fn` recording how it was created

PyTorch builds this graph dynamically during the forward pass. `.backward()` traverses it in reverse, computing gradients at every compute node and accumulating them at the parameter nodes.

In [7]:
# 3 instances, 2 input features — minimal realistic case
# Same decomposition as Session 4:
#   (1) yhat = X @ theta_1.T + theta_0
#   (2) z    = yhat - y
#   (3) L    = mean(z ** 2)

# parameters
theta_1 = torch.tensor([[1.0, 0.5]], requires_grad=True)  # shape (1, 2) — weight
theta_0 = torch.tensor([[0.2]],      requires_grad=True)  # shape (1, 1) — bias

# data — 3 instances, 2 features
X = torch.tensor([[1.0, 2.0],
                   [3.0, 0.5],
                   [2.0, 1.5]])           # shape (3, 2)
y = torch.tensor([[3.0], [2.0], [4.0]])  # shape (3, 1)

print('--- Parameters (requires_grad=True) ---')
for name, t in [('theta_1', theta_1), ('theta_0', theta_0)]:
    print(f'  {name}: shape={list(t.shape)}, requires_grad={t.requires_grad}, is_leaf={t.is_leaf}')

print('\n--- Data (requires_grad=False) ---')
for name, t in [('X', X), ('y', y)]:
    print(f'  {name}: shape={list(t.shape)}, requires_grad={t.requires_grad}, is_leaf={t.is_leaf}')
print('  -> gradients will NOT be computed for data tensors')

# forward pass
yhat = X @ theta_1.T + theta_0    # (1) shape (3, 1)
z    = yhat - y                    # (2) shape (3, 1)
L    = (z ** 2).mean()             # (3) scalar

print('\n--- Compute nodes ---')
for name, t in [('yhat', yhat), ('z', z), ('L', L)]:
    print(f'  {name}: shape={list(t.shape)}, grad_fn={t.grad_fn.__class__.__name__}, is_leaf={t.is_leaf}')

# backward
L.backward()
print('\n--- After backward ---')
print(f'  theta_1.grad: {theta_1.grad}  shape={list(theta_1.grad.shape)}')
print(f'  theta_0.grad: {theta_0.grad}  shape={list(theta_0.grad.shape)}')
print(f'  X.grad:       {X.grad}')       # None — data
print(f'  z.grad:       {z.grad}')       # None — compute node, not leaf

--- Parameters (requires_grad=True) ---
  theta_1: shape=[1, 2], requires_grad=True, is_leaf=True
  theta_0: shape=[1, 1], requires_grad=True, is_leaf=True

--- Data (requires_grad=False) ---
  X: shape=[3, 2], requires_grad=False, is_leaf=True
  y: shape=[3, 1], requires_grad=False, is_leaf=True
  -> gradients will NOT be computed for data tensors

--- Compute nodes ---
  yhat: shape=[3, 1], grad_fn=AddBackward0, is_leaf=False
  z: shape=[3, 1], grad_fn=SubBackward0, is_leaf=False
  L: shape=[], grad_fn=MeanBackward0, is_leaf=False

--- After backward ---
  theta_1.grad: tensor([[ 0.9667, -1.6333]])  shape=[1, 2]
  theta_0.grad: tensor([[-0.2667]])  shape=[1, 1]
  X.grad:       None
  z.grad:       None


/tmp/ipykernel_17668/3969803527.py:41: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(f'  z.grad:       {z.grad}')       # None — compute node, not leaf


<!-- FULL: keep, DEMO: delete -->
By default, input data tensors have `requires_grad=False` — gradients with respect to inputs are not computed. But setting `requires_grad=True` on an input tensor makes PyTorch compute $\partial L / \partial x$ as well.

This is useful in several contexts:

- **Explainability / saliency maps** — which input features most influence the output? The gradient $\partial L / \partial x_j$ tells you how sensitive the loss is to each input dimension
- **Adversarial examples** — perturbing inputs in the gradient direction to fool the network
- **Style transfer and image generation** — optimising the input image rather than the parameters

In [8]:
# set requires_grad=True on input to get gradient w.r.t. data
X_explain = torch.tensor([[1.0, 2.0],
                           [3.0, 0.5],
                           [2.0, 1.5]], requires_grad=True)  # shape (3, 2)
y_fixed   = torch.tensor([[3.0], [2.0], [4.0]])

# fixed trained parameters
t1 = theta_1.detach()
t0 = theta_0.detach()

yhat = X_explain @ t1.T + t0
z    = yhat - y_fixed
L    = (z ** 2).mean()
L.backward()

print(f'dL/dX shape: {X_explain.grad.shape}')   # (3, 2) — one gradient per input value
print(f'dL/dX:\n{X_explain.grad}')
print()
print('Each row: gradient for one instance')
print('Each column: sensitivity of loss to that feature')
print(f'\nFeature importance (mean |grad| across instances):')
print(f'  feature 0: {X_explain.grad[:, 0].abs().mean().item():.4f}')
print(f'  feature 1: {X_explain.grad[:, 1].abs().mean().item():.4f}')

dL/dX shape: torch.Size([3, 2])
dL/dX:
tensor([[-0.5333, -0.2667],
        [ 0.9667,  0.4833],
        [-0.7000, -0.3500]])

Each row: gradient for one instance
Each column: sensitivity of loss to that feature

Feature importance (mean |grad| across instances):
  feature 0: 0.7333
  feature 1: 0.3667


In [9]:
print(f'yhat.grad_fn: {yhat.grad_fn}')   # AddmmBackward (matmul + bias)
print(f'z.grad_fn:    {z.grad_fn}')        # SubBackward
print(f'L.grad_fn:    {L.grad_fn}')         # MeanBackward

print(f'\nL.grad_fn.next_functions:')
for fn in L.grad_fn.next_functions:
    print(f'  {fn}')

yhat.grad_fn: <AddBackward0 object at 0x77626d6ebbe0>
z.grad_fn:    <SubBackward0 object at 0x77626d52ece0>
L.grad_fn:    <MeanBackward0 object at 0x77626d52ece0>

L.grad_fn.next_functions:
  (<PowBackward0 object at 0x77626cf79e40>, 0)


<!-- FULL: keep, DEMO: delete -->
**`.detach()` — cutting a tensor out of the graph:**

A detached tensor has no `grad_fn` — it is treated as a constant by autograd. Gradients do not flow through it during `.backward()`.

Practical use: **logging the loss** without keeping the computation graph alive. Calling `.item()` also detaches, but `.detach()` is explicit and works for tensors of any shape.

In [10]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
z     = theta[1] * 2.0 + theta[0]

print(f'z.grad_fn:           {z.grad_fn}')         # AddBackward — in graph

z_detached = z.detach()
print(f'z_detached.grad_fn:  {z_detached.grad_fn}') # None — cut from graph
print(f'z_detached value:    {z_detached}')          # same value, no history

# backward still works through the original z
loss = z ** 2
loss.backward()
print(f'\ntheta.grad:          {theta.grad}')         # gradients computed correctly

z.grad_fn:           <AddBackward0 object at 0x77626d647760>
z_detached.grad_fn:  None
z_detached value:    2.0

theta.grad:          tensor([4., 8.])


In [11]:
# Common pattern: log loss value without retaining graph
loss_history = []

theta = torch.tensor([1.0, 0.5], requires_grad=True)
for step in range(4):
    if theta.grad is not None: theta.grad.zero_()
    z    = theta[1] * 2.0 + theta[0]
    loss = z ** 2
    loss_history.append(loss.item())   # .item() detaches scalar to Python float
    loss.backward()
    with torch.no_grad():
        theta -= 0.1 * theta.grad

print('Loss history:', [f'{l:.4f}' for l in loss_history])

Loss history: ['4.0000', '0.0000', '0.0000', '0.0000']


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>.backward()</code> and <code>.grad</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Calling `.backward()` on a scalar loss traverses the graph in reverse, 
computing $\partial L / \partial \theta$ for every leaf tensor with `requires_grad=True`. 
The result is stored in `.grad`.

In [12]:
# fresh tensors for clean backward demo
theta_1 = torch.tensor([[1.0, 0.5]], requires_grad=True)
theta_0 = torch.tensor([[0.2]],      requires_grad=True)
X = torch.tensor([[1.0, 2.0], [3.0, 0.5], [2.0, 1.5]])
y = torch.tensor([[3.0], [2.0], [4.0]])

yhat = X @ theta_1.T + theta_0
z    = yhat - y
L    = (z ** 2).mean()
L.backward()

print(f'theta_1.grad: {theta_1.grad}')   # dL/d(theta_1) shape (1, 2)
print(f'theta_0.grad: {theta_0.grad}')   # dL/d(theta_0) shape (1, 1)

# manual check: dL/d(theta_1) = (2/N) * z.T @ X
N = X.shape[0]
manual_t1 = (2 / N) * z.T @ X
manual_t0 = (2 / N) * z.sum(dim=0, keepdim=True)
print(f'\nManual dL/d(theta_1): {manual_t1}')
print(f'Match: {torch.allclose(theta_1.grad, manual_t1)}')
print(f'Manual dL/d(theta_0): {manual_t0}')
print(f'Match: {torch.allclose(theta_0.grad, manual_t0)}')

theta_1.grad: tensor([[ 0.9667, -1.6333]])
theta_0.grad: tensor([[-0.2667]])

Manual dL/d(theta_1): tensor([[ 0.9667, -1.6333]], grad_fn=<MmBackward0>)
Match: True
Manual dL/d(theta_0): tensor([[-0.2667]], grad_fn=<MulBackward0>)
Match: True


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; calling .backward() twice
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch frees the computation graph after `.backward()` to save memory. 
Calling it again raises a `RuntimeError`.

If you need to call `.backward()` multiple times (e.g. for higher-order gradients), 
pass `retain_graph=True` — but this is rarely needed in standard training.

In [14]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
loss = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()   # first call — fine
loss.backward()   # second call — RuntimeError!

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Why <code>.zero_grad()</code>?
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch **accumulates** gradients — each `.backward()` call **adds** to `.grad`. 
Always zero gradients before each backward pass in a training loop.

In [15]:
# gradients accumulate across backward() calls
theta = torch.tensor([1.0, 0.5], requires_grad=True)

for step in range(3):
    z    = theta[1] * x[0] + theta[0]
    loss = z ** 2
    loss.backward()
    print(f'step {step}: theta.grad = {theta.grad}')  # keeps growing!

step 0: theta.grad = tensor([3., 3.])
step 1: theta.grad = tensor([6., 6.])
step 2: theta.grad = tensor([9., 9.])


In [16]:
# correct: zero before each backward
theta = torch.tensor([1.0, 0.5], requires_grad=True)

for step in range(3):
    if theta.grad is not None:
        theta.grad.zero_()
    z    = theta[1] * x[0] + theta[0]
    loss = z ** 2
    loss.backward()
    print(f'step {step}: theta.grad = {theta.grad}')

step 0: theta.grad = tensor([3., 3.])
step 1: theta.grad = tensor([3., 3.])
step 2: theta.grad = tensor([3., 3.])


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; zeroing gradients
</div>

<!-- FULL: keep, DEMO: delete -->
Three ways to zero gradients — all correct, with subtle differences:

- `tensor.grad.zero_()` — zeros in-place, keeps the tensor allocated
- `tensor.grad = None` — releases memory; slightly faster if grad not always needed
- `optimizer.zero_grad()` — preferred in training loops; handles all params at once
- `optimizer.zero_grad(set_to_none=True)` — sets to `None` instead of zero; default in PyTorch ≥ 2.0

In [17]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
loss  = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
print(f'Before: theta.grad = {theta.grad}')

# Option 1 — in-place zero
theta.grad.zero_()
print(f'After zero_():      {theta.grad}')

# Option 2 — set to None (re-run backward first)
loss = (theta[1] * 2.0 + theta[0]) ** 2
loss.backward()
theta.grad = None
print(f'After grad=None:    {theta.grad}')

Before: theta.grad = tensor([4., 8.])
After zero_():      tensor([0., 0.])
After grad=None:    None


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; in-place operations on requires_grad tensors
</div>

<!-- FULL: keep, DEMO: delete -->
In-place operations (e.g. `+=`, `.fill_()`, `.relu_()`) modify a tensor's data 
without creating a new node in the graph. This corrupts the graph PyTorch 
needs for backpropagation and raises a `RuntimeError`.

In [18]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)
z = theta * 2.0

theta += 1.0   # in-place modification — RuntimeError!

RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>torch.no_grad()</code>
</div>

<!-- FULL: keep, DEMO: delete -->
During evaluation, gradients are not needed. 
`torch.no_grad()` disables graph construction — saves memory and computation.

In [19]:
theta = torch.tensor([1.0, 0.5], requires_grad=True)

# with grad tracking (default)
z = theta[1] * x[0] + theta[0]
print(f'grad tracking on:  z.grad_fn = {z.grad_fn}')

# without grad tracking
with torch.no_grad():
    z = theta[1] * x[0] + theta[0]
    print(f'grad tracking off: z.grad_fn = {z.grad_fn}')

grad tracking on:  z.grad_fn = <AddBackward0 object at 0x77626cecbe50>
grad tracking off: z.grad_fn = None


<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>2</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Building Networks &mdash; <code>nn.Module</code></span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Why <code>nn.Module</code>?
</div>

<!-- FULL: keep, DEMO: delete -->
As networks grow deeper, managing raw parameter tensors becomes error-prone:
tracking all weights and biases, passing them to the optimiser, switching train/eval behaviour.

`nn.Module` handles parameter registration, device placement, and train/eval switching automatically.

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; forgetting super().__init__()
</div>

<!-- FULL: keep, DEMO: delete -->
Every `nn.Module` subclass must call `super().__init__()` as the first line of `__init__`. 
Without it, PyTorch's parameter registration machinery is not set up and you get a cryptic `AttributeError`.

In [ ]:
class BrokenModel(nn.Module):
    def __init__(self):
        # missing: super().__init__()
        self.linear = nn.Linear(4, 1)   # AttributeError!

BrokenModel()

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Custom <code>nn.Module</code> — minimal example
</div>

<!-- FULL: keep, DEMO: delete -->
Subclass `nn.Module`, implement `__init__` (register layers) and `forward` (computation). Nothing else required.

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module):
    """Single linear layer: y = X @ theta_1.T + theta_0."""

    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = LinearModel(in_features=4, out_features=1)
print(model)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>nn.Linear</code> and parameters
</div>

<!-- FULL: keep, DEMO: delete -->
`nn.Linear(in, out)` stores `weight` (shape `[out, in]`) and `bias` (shape `[out]`) 
as `nn.Parameter` objects — automatically tracked by `model.parameters()`.

Correspondence with our notation: `weight` $\leftrightarrow$ $\Theta_1$, `bias` $\leftrightarrow$ $\Theta_0$.

In [ ]:
# inspect parameters
for name, param in model.named_parameters():
    print(f'{name:30s}  shape={str(param.shape):15s}  requires_grad={param.requires_grad}')

In [ ]:
# correspondence with our manual notation:
# nn.Linear stores:  weight  shape (out, in)  <->  theta_1
#                    bias    shape (out,)      <->  theta_0
print(f'weight (theta_1): {model.linear.weight.shape}')
print(f'bias   (theta_0): {model.linear.bias.shape}')

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  MLP — <code>nn.Module</code> with multiple layers
</div>

In [ ]:
class MLP(nn.Module):
    """Two-layer MLP: Linear -> ReLU -> Linear."""

    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x


mlp = MLP(in_features=4, hidden=8, out_features=1)
print(mlp)
print(f'\nTotal parameters: {sum(p.numel() for p in mlp.parameters())}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; defining an MLP — three styles
</div>

<!-- FULL: keep, DEMO: delete -->
All three produce equivalent networks. Choice depends on flexibility needed:

- **Explicit attributes** — most readable, easiest to debug
- **`nn.ModuleList`** — useful when number of layers varies at runtime
- **`nn.Sequential`** — most concise for simple feed-forward networks

In [ ]:
# Style 1 — explicit named attributes
class MLP_v1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 8)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(8, 1)
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

# Style 2 — nn.ModuleList with loop
class MLP_v2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)])
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return x

# Style 3 — nn.Sequential
mlp_v3 = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

# All have the same parameter count
for name, m in [('v1', MLP_v1()), ('v2', MLP_v2()), ('v3', mlp_v3)]:
    print(f'{name}: {sum(p.numel() for p in m.parameters())} parameters')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>nn.Sequential</code> shortcut
</div>

<!-- FULL: keep, DEMO: delete -->
For simple feed-forward networks, `nn.Sequential` avoids subclassing — list layers in order.

In [ ]:
mlp_seq = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
print(mlp_seq)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; using a plain list instead of nn.ModuleList
</div>

<!-- FULL: keep, DEMO: delete -->
Layers stored in a plain Python list are **not** registered as submodules. 
`model.parameters()` will not see them — they will not be updated during training.

In [ ]:
class BrokenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # plain list — parameters not registered!
        self.layers = [nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)]

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

broken = BrokenMLP()
print('Parameters found:', list(broken.parameters()))   # empty!

# Fix: use nn.ModuleList
class FixedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

fixed = FixedMLP()
print('Parameters found:', sum(p.numel() for p in fixed.parameters()))  # correct

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>model.train()</code> and <code>model.eval()</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Some layers (Dropout, BatchNorm — Session 7) behave differently in train vs eval.
Always call `model.eval()` before validation and `model.train()` before resuming training.

In [ ]:
print(f'Default:      training={mlp.training}')
mlp.eval()
print(f'After eval(): training={mlp.training}')
mlp.train()
print(f'After train(): training={mlp.training}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>3</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Optimisers &mdash; <code>torch.optim</code></span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>torch.optim</code> — overview
</div>

<!-- FULL: keep, DEMO: delete -->
All optimisers from Session 5 are available in `torch.optim`.
Each takes `model.parameters()` and implements the update rule.

Update sequence — always in this order:
1. `optimizer.zero_grad()` &nbsp; clear accumulated gradients
2. `loss.backward()` &nbsp; compute gradients
3. `optimizer.step()` &nbsp; apply update rule

In [ ]:
import torch.optim as optim

# SGD
optimizer_sgd = optim.SGD(mlp.parameters(), lr=0.01)

# SGD with momentum
optimizer_momentum = optim.SGD(mlp.parameters(), lr=0.01, momentum=0.9)

# Adam
optimizer_adam = optim.Adam(mlp.parameters(), lr=1e-3)

print('Optimisers created.')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; optimiser options
</div>

<!-- FULL: keep, DEMO: delete -->
All optimisers from Session 5 are available. Per-layer learning rates are supported via **param groups**.

In [ ]:
# Standard usage
opt_sgd      = optim.SGD(mlp.parameters(),  lr=0.01)
opt_momentum = optim.SGD(mlp.parameters(),  lr=0.01, momentum=0.9, weight_decay=1e-4)
opt_adam     = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
opt_adamw    = optim.AdamW(mlp.parameters(), lr=1e-3)  # Adam with decoupled weight decay

# Per-layer learning rates via param groups
opt_perlayer = optim.Adam([
    {'params': mlp.layer1.parameters(), 'lr': 1e-4},
    {'params': mlp.layer2.parameters(), 'lr': 1e-3},
])
print('Optimisers created.')

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  One optimiser step — in detail
</div>

In [ ]:
# load data
import sys
sys.path.append('../../A2/')
from helpers import load_data
X, y = load_data('../../A2/data.csv')

torch.manual_seed(0)
mlp   = MLP(in_features=4, hidden=8, out_features=1)
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

# before update
w_before = mlp.layer1.weight.data.clone()
print(f'layer1 weight before: {w_before[0, :3]}')

# one step
optimizer.zero_grad()          # 1. clear gradients
y_pred = mlp(X)                # 2. forward pass
loss   = loss_fn(y_pred, y)    # 3. compute loss
loss.backward()                # 4. backward pass
optimizer.step()               # 5. update parameters

w_after = mlp.layer1.weight.data.clone()
print(f'layer1 weight after:  {w_after[0, :3]}')
print(f'loss: {loss.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; loss functions — nn vs functional
</div>

<!-- FULL: keep, DEMO: delete -->
PyTorch provides loss functions in two equivalent forms:

- `nn.MSELoss()` — stateful object, can be stored as an attribute
- `F.mse_loss()` — stateless function from `torch.nn.functional`; more flexible

Both compute the same result. `nn` style is more common in training loops; `F` style is common inside `forward()`.

In [ ]:
import torch.nn.functional as F

y_pred = torch.tensor([[1.5], [2.0], [0.5]])
y_true = torch.tensor([[1.0], [2.5], [0.3]])

# nn style
loss_nn = nn.MSELoss()(y_pred, y_true)

# functional style
loss_f  = F.mse_loss(y_pred, y_true)

print(f'nn.MSELoss:  {loss_nn.item():.4f}')
print(f'F.mse_loss:  {loss_f.item():.4f}')
print(f'Identical:   {torch.allclose(loss_nn, loss_f)}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>4</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Data Pipeline &mdash; <code>Dataset</code> &amp; <code>DataLoader</code></span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Custom <code>Dataset</code>
</div>

<!-- FULL: keep, DEMO: delete -->
Subclass `torch.utils.data.Dataset` and implement:
- `__len__`: number of examples
- `__getitem__`: return the $i$-th example

Works for data too large to fit in memory, data loaded from disk, or on-the-fly augmentation.

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

class HousingDataset(Dataset):
    """Housing price dataset."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = HousingDataset(X, y)
print(f'Dataset size: {len(dataset)}')
x0, y0 = dataset[0]
print(f'First example: x={x0}, y={y0.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>TensorDataset</code> shortcut
</div>

<!-- FULL: keep, DEMO: delete -->
When data fits in memory as tensors, `TensorDataset` wraps them without subclassing.

In [ ]:
dataset = TensorDataset(X, y)
print(f'TensorDataset size: {len(dataset)}')
x0, y0 = dataset[0]
print(f'First example: x shape={x0.shape}, y={y0.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Train / validation split
</div>

<!-- FULL: keep, DEMO: delete -->
`random_split` divides a dataset into non-overlapping subsets reproducibly.

In [ ]:
torch.manual_seed(0)
n_train = int(0.8 * len(dataset))
n_val   = len(dataset) - n_train
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

print(f'Train size:      {len(train_dataset)}')
print(f'Validation size: {len(val_dataset)}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  <code>DataLoader</code> — batching and shuffling
</div>

<!-- FULL: keep, DEMO: delete -->
`DataLoader` wraps a `Dataset` and handles mini-batching, shuffling, and parallel loading.

Always `shuffle=True` for training, `shuffle=False` for validation.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f'Batches per epoch (train): {len(train_loader)}')
print(f'Batches per epoch (val):   {len(val_loader)}')

# inspect one batch
X_batch, y_batch = next(iter(train_loader))
print(f'\nBatch shapes: X={X_batch.shape}, y={y_batch.shape}')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(0,85,100); background:rgb(235,245,247); color:rgb(0,85,100); font-size:1.1em; font-weight:bold;'>
  &#8644; Alternatives &mdash; train / validation split approaches
</div>

<!-- FULL: keep, DEMO: delete -->
- `random_split` — simple, built-in, reproducible with `torch.manual_seed`
- Manual index slicing — full control over which examples go where
- `sklearn.model_selection.train_test_split` — more options (stratified, grouped)

In [ ]:
# Option 1 — random_split (already shown)
train_ds, val_ds = random_split(dataset, [0.8, 0.2],
                                generator=torch.Generator().manual_seed(0))

# Option 2 — manual index split
n       = len(dataset)
indices = torch.randperm(n, generator=torch.Generator().manual_seed(0))
n_train = int(0.8 * n)
train_ds2 = torch.utils.data.Subset(dataset, indices[:n_train])
val_ds2   = torch.utils.data.Subset(dataset, indices[n_train:])

print(f'random_split:  train={len(train_ds)}, val={len(val_ds)}')
print(f'manual split:  train={len(train_ds2)}, val={len(val_ds2)}')

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>5</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Full Training Loop</span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Training and validation functions
</div>

<!-- FULL: keep, DEMO: delete -->
Separate training and validation logic into reusable functions.
Note `model.train()` / `model.eval()` and `torch.no_grad()` in the validation loop.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """One epoch of training. Returns mean loss."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def val_epoch(model, loader, loss_fn):
    """One epoch of validation. Returns mean loss."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred     = model(X_batch)
            loss       = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


print('Functions defined.')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; forgetting model.eval() and torch.no_grad() in validation
</div>

<!-- FULL: keep, DEMO: delete -->
Without `model.eval()`, layers like Dropout and BatchNorm (Session 7) behave as if training — 
validation metrics will be wrong.

Without `torch.no_grad()`, PyTorch builds and retains the computation graph for every validation batch — 
memory grows unnecessarily and validation is slower.

In [ ]:
# Wrong — no eval(), no no_grad()
def val_epoch_broken(model, loader, loss_fn):
    total_loss = 0.0
    for X_batch, y_batch in loader:
        y_pred     = model(X_batch)          # graph built unnecessarily
        loss       = loss_fn(y_pred, y_batch)
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

# Correct — always use both
def val_epoch_correct(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred     = model(X_batch)
            loss       = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Training MLP on housing data
</div>

In [ ]:
torch.manual_seed(0)
model     = MLP(in_features=4, hidden=16, out_features=1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

n_epochs    = 100
train_losses = []
val_losses   = []

for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    val_loss   = val_epoch(model, val_loader, loss_fn)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}  train loss: {train_loss:.4f}  val loss: {val_loss:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, color='#005564', label='train loss')
ax.plot(val_losses,   color='#FF6A00', label='val loss')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Training curve')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# cleaner plotting using helpers
from plotting import plot_losses
plot_losses(train_losses, val_losses)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Reading loss curves
</div>

<!-- FULL: keep, DEMO: delete -->
- **Both losses decrease** — training is working
- **Train loss $\ll$ val loss** — overfitting; model memorises training data
- **Val loss stops improving** — stop training or regularise
- **Loss spikes suddenly** — learning rate too large

Overfitting and regularisation covered in Session 7.

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'>6</span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Going Deeper</span>
</div>

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Configurable <code>DeepMLP</code>
</div>

<!-- FULL: keep, DEMO: delete -->
With `nn.Module` subclassing, depth and width become constructor arguments.
`nn.Sequential(*layers)` assembles any list of layers dynamically.

In [ ]:
class DeepMLP(nn.Module):
    """MLP with configurable depth and width."""

    def __init__(self, in_features, hidden_sizes, out_features):
        """
        Args:
            in_features:  number of input features
            hidden_sizes: list of hidden layer widths, e.g. [32, 16, 8]
            out_features: number of outputs
        """
        super().__init__()
        layers = []
        sizes  = [in_features] + hidden_sizes
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# shallow
shallow = DeepMLP(4, [8], 1)
# deep
deep    = DeepMLP(4, [32, 16, 8], 1)

print('Shallow:', sum(p.numel() for p in shallow.parameters()), 'parameters')
print(shallow)
print('\nDeep:   ', sum(p.numel() for p in deep.parameters()), 'parameters')
print(deep)

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Shallow vs deep — convergence comparison
</div>

In [ ]:
def train_model(model, train_loader, val_loader, n_epochs=100, lr=1e-3):
    """Train a model and return train/val loss histories."""
    optimizer    = optim.Adam(model.parameters(), lr=lr)
    loss_fn      = nn.MSELoss()
    train_losses = []
    val_losses   = []
    for _ in range(n_epochs):
        train_losses.append(train_epoch(model, train_loader, loss_fn, optimizer))
        val_losses.append(val_epoch(model, val_loader, loss_fn))
    return train_losses, val_losses


torch.manual_seed(0)
shallow = DeepMLP(4, [8], 1)
tl_s, vl_s = train_model(shallow, train_loader, val_loader)

torch.manual_seed(0)
deep = DeepMLP(4, [32, 16, 8], 1)
tl_d, vl_d = train_model(deep, train_loader, val_loader)

print(f'Shallow — final val loss: {vl_s[-1]:.4f}')
print(f'Deep    — final val loss: {vl_d[-1]:.4f}')

In [ ]:
from plotting import plot_losses_compare
plot_losses_compare(
    [(tl_s, vl_s, 'shallow [8]'),
     (tl_d, vl_d, 'deep [32,16,8]')]
)

<!-- FULL: keep, DEMO: keep -->
<br/>

<!-- FULL: keep, DEMO: keep -->
<div style='margin:3em 0 1.5em 0; padding:0.8em 1.2em; background:rgb(22,60,105); border-radius:6px; display:flex; align-items:center; gap:1em;'>
  <span style='color:rgb(255,106,0); font-size:2em; font-weight:bold; line-height:1;'></span>
  <span style='color:white; font-size:1.4em; font-weight:bold;'>Summary</span>
</div>

<!-- FULL: keep, DEMO: keep -->
| Concept | PyTorch API |
|---|---|
| Automatic differentiation | `requires_grad`, `.backward()`, `.grad` |
| Network definition | `nn.Module` subclass, `nn.Linear`, `nn.ReLU` |
| Parameter access | `model.parameters()`, `model.named_parameters()` |
| Optimisation | `torch.optim.SGD`, `torch.optim.Adam` |
| Data loading | `Dataset`, `TensorDataset`, `DataLoader` |
| Train/eval modes | `model.train()`, `model.eval()`, `torch.no_grad()` |

<div style='margin-top:1em; color:rgb(0,85,100);'><b>Next:</b> Session 7 — Regularisation &amp; Generalisation</div>